In [1]:
from gtf_utils import filter_gtf_by_features, keep_only_main_gtf_transcripts, filter_gtf_by_transcripts, sort_gtf, index_gtf, calculate_cds_pos, gz_to_bgz
import math
import pyranges1 as pr
import pandas as pd
import subprocess
from pathlib import Path
import gzip
import pysam
from contextlib import nullcontext

In [2]:
pd.set_option('display.max_columns', None)

## GFF preparation + BED preparation

In [ ]:
gtf_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.gtf.gz'

In [16]:
command = f'''./prepare_data -i {gtf_filepath} -rm 1'''
subprocess.run(command, shell=True, check=True)

CompletedProcess(args='./prepare_data -i /Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.gtf.gz -rm 1', returncode=0)

## Test

In [ ]:
gtf_cds_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.cds.gtf.gz'
cds_pr = pr.read_gtf(gtf_cds_filepath, duplicate_attr=True)

In [ ]:
cds_table_calculated = calculate_cds_pos(cds_pr)
cds_table_calculated_short = cds_table_calculated[['Chromosome', 'Start', 'End', 'Strand', 'transcript_id', 'cds_start', 'cds_end']]

In [ ]:
chrom = 'chr1'
pos = int('12861477')

In [ ]:
test_pr = cds_table_calculated_short.query('@chrom == Chromosome and @pos >= Start and @pos <= End')

In [ ]:
# Ограничение: для транскриптов с неполным началом CDS (cds_start_NF) позиции рассчитываются от первого доступного нуклеотида и могут не соответствовать позициям в полном белке.
# cds_start_NF (CDS start not found) — тег, обозначающий, что начало кодирующей последовательности транскрипта не определено полностью. Первый аннотированный участок CDS может начинаться внутри кодона, а не со стартового кодона.
pos_zero_based = pos - 1

for row in test_pr.itertuples(index=False):
    if row.Chromosome == chrom and row.Start <= pos_zero_based < row.End:
        if row.Strand == '+':
            offset = pos_zero_based - row.Start
        else:
            offset = row.End - 1 - pos_zero_based

        variant_cds_pos = row.cds_start + offset
        variant_protein_pos = (variant_cds_pos - 1) // 3 + 1

        print(row.transcript_id, variant_cds_pos, variant_protein_pos)

In [ ]:
pd.DataFrame(cds_pr)

## MNV calculation from VCF

In [ ]:
input_vcf_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/test_files/vcf/000007239270_DeepVariant.vcf.gz'
fasta_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.fa'
bam_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/test_files/bam/000007239270.bam.genozip'
reference_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.ref.genozip'
bed_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/cds.bed'
output_vcf_filepath = input_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.mnv.vcf.gz'

command = f'''./cis2mnv -i {input_vcf_filepath} -f {fasta_filepath} -b {bam_genozip_filepath} -gr {reference_genozip_filepath} --bed {bed_filepath} -rm 1 -c 1 -o {output_vcf_filepath}'''
subprocess.run(command, shell=True, check=True)

## VCF annotation

### VEP

In [ ]:
gtf_single_transcripts_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.single_transcripts.sorted.gtf.gz'
vep_annotated_vcf_filepath = output_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.vep.vcf.gz'

command = f'''mamba run -n cis2mnv-vep.venv vep \
    --input_file {output_vcf_filepath} \
    --output_file {vep_annotated_vcf_filepath} \
    --gtf {gtf_single_transcripts_filepath} \
    --fasta {fasta_filepath} \
    --warning_file STDERR \
    --no_stats \
    --force_overwrite \
    --format vcf \
    --vcf \
    --compress_output bgzip \
    --symbol \
    --protein \
    --hgvs'''
subprocess.run(command, shell=True, check=True)

### SnpEff

In [ ]:
snpeff_annotated_vcf_filepath = output_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.snpeff.vcf.gz'

command = f'''mamba run -n cis2mnv-snpeff.venv snpeff ann -canon -noStats -download hg38 {output_vcf_filepath} | bgzip -c > {snpeff_annotated_vcf_filepath}'''
subprocess.run(command, shell=True, check=True)